# Choosing the Right Pattern

You have a catalog of agent patterns — routing, ReAct, plan-and-execute, reflection, and more. **Choosing the right one** means matching **problem signals** to the **simplest pattern that meets requirements**, then validating with evaluation before shipping.

```
  Requirements + constraints
         │
         ▼
  ┌──────────────────┐
  │  Signal scoring  │  static FAQ? fixed SOP? dynamic tools?
  └────────┬─────────┘
           ▼
  ┌──────────────────┐
  │ Pattern shortlist│  top 1–2 candidates
  └────────┬─────────┘
           ▼
  ┌──────────────────┐
  │  Eval harness    │  escalate only if simpler tier fails
  └────────┬─────────┘
           ▼
     Architecture decision record
```

This notebook provides a **decision framework**, a **scoring selector**, worked examples for **ShopCo Support Hub** and **ReleaseFlow**, pattern **combinations**, and an **ADR template** you can paste into design docs.

In [ ]:
"""
Pattern selection lab — ShopCo + ReleaseFlow.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days. [pol-returns-01]",
    "shipping": "Free shipping over $50. [pol-shipping-02]",
    "hours": "Support hours Mon–Fri 9am–6pm ET. [faq-hours-05]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing"},
}

RELEASE_CHECKS = ["unit_tests", "lint", "security_scan"]

print("Pattern selection lab ready.")

---

## 1. Decision Questions (Ask Before Coding)

| # | Question | Points toward |
|---|----------|---------------|
| 1 | Is the answer deterministic from a lookup table? | **Lookup** |
| 2 | Are steps identical for every request? | **Prompt chain** |
| 3 | Are intents disjoint with different handlers? | **Routing** |
| 4 | Is one tool call + template enough? | **One-shot tool** |
| 5 | Is tool order unknown at design time? | **ReAct** |
| 6 | Is there a multi-step SOP to plan then execute? | **Plan-and-execute** |
| 7 | Can subtasks run independently? | **Parallelization** |
| 8 | Does a lead delegate dynamic subtasks? | **Orchestrator-workers** |
| 9 | Is output quality graded by a rubric score? | **Evaluator-optimizer** |
| 10 | Are fixes qualitative (tone, sections)? | **Reflection** |
| 11 | Multiple roles + shared board + QA? | **Multi-agent collaboration** |

---

## 2. Pattern Registry — Signals and Anti-Signals

In [ ]:
class AgentPattern(str, Enum):
    LOOKUP = "lookup"
    PROMPT_CHAINING = "prompt_chaining"
    ROUTING = "routing"
    ONE_SHOT_TOOL = "one_shot_tool"
    PARALLELIZATION = "parallelization"
    ORCHESTRATOR_WORKERS = "orchestrator_workers"
    REACT = "react"
    PLAN_AND_EXECUTE = "plan_and_execute"
    EVALUATOR_OPTIMIZER = "evaluator_optimizer"
    REFLECTION = "reflection"
    MULTI_AGENT = "multi_agent"


@dataclass
class PatternProfile:
    pattern: AgentPattern
    name: str
    signals: list[str]
    anti_signals: list[str]
    complexity: int  # 1–5
    typical_llm_calls: int


PATTERN_REGISTRY: list[PatternProfile] = [
    PatternProfile(AgentPattern.LOOKUP, "Lookup", ["static faq", "deterministic"], ["personalized", "multi-tool"], 1, 0),
    PatternProfile(AgentPattern.PROMPT_CHAINING, "Prompt Chaining", ["fixed steps", "etl text"], ["dynamic branch"], 2, 2),
    PatternProfile(AgentPattern.ROUTING, "Routing", ["disjoint intents", "specialists"], ["unknown tool order"], 2, 2),
    PatternProfile(AgentPattern.ONE_SHOT_TOOL, "One-Shot Tool", ["single lookup"], ["multi-step sop"], 2, 1),
    PatternProfile(AgentPattern.PARALLELIZATION, "Parallelization", ["independent checks", "fan-out"], ["strict sequence"], 3, 1),
    PatternProfile(AgentPattern.ORCHESTRATOR_WORKERS, "Orchestrator-Workers", ["multi-domain", "dynamic subtasks"], ["single intent"], 4, 4),
    PatternProfile(AgentPattern.REACT, "ReAct", ["exploratory tools", "unknown sequence"], ["fixed sop"], 4, 4),
    PatternProfile(AgentPattern.PLAN_AND_EXECUTE, "Plan-and-Execute", ["reviewable plan", "long horizon"], ["two-step fixed"], 4, 3),
    PatternProfile(AgentPattern.EVALUATOR_OPTIMIZER, "Evaluator-Optimizer", ["numeric rubric", "threshold"], ["pure lookup"], 3, 4),
    PatternProfile(AgentPattern.REFLECTION, "Reflection", ["tone", "qualitative critique"], ["numeric-only gate"], 3, 3),
    PatternProfile(AgentPattern.MULTI_AGENT, "Multi-Agent", ["separate contracts", "qa gate"], ["simple faq"], 5, 6),
]

print("Patterns:", [p.name for p in PATTERN_REGISTRY])

---

## 3. Requirement Profile — Encode the Problem

In [ ]:
@dataclass
class RequirementProfile:
    task_id: str
    description: str
    static_faq: bool = False
    fixed_steps: bool = False
    disjoint_intents: bool = False
    single_tool: bool = False
    dynamic_tools: bool = False
    multi_step_sop: bool = False
    parallel_checks: bool = False
    multi_domain: bool = False
    numeric_quality_gate: bool = False
    qualitative_polish: bool = False
    separate_agent_roles: bool = False
    latency_budget_ms: int = 3000
    human_plan_review: bool = False

    def active_signals(self) -> list[str]:
        mapping = {
            "static faq": self.static_faq,
            "fixed steps": self.fixed_steps,
            "disjoint intents": self.disjoint_intents,
            "single lookup": self.single_tool,
            "exploratory tools": self.dynamic_tools,
            "reviewable plan": self.multi_step_sop or self.human_plan_review,
            "long horizon": self.multi_step_sop,
            "independent checks": self.parallel_checks,
            "multi-domain": self.multi_domain,
            "numeric rubric": self.numeric_quality_gate,
            "tone": self.qualitative_polish,
            "separate contracts": self.separate_agent_roles,
        }
        return [k for k, v in mapping.items() if v]


shopco_return = RequirementProfile(
    task_id="shopco-return",
    description="Return eligibility needs policy + order lookup",
    dynamic_tools=True,
    qualitative_polish=True,
)
print("Signals:", shopco_return.active_signals())

---

## 4. Pattern Selector — Weighted Scoring

In [ ]:
@dataclass
class PatternRecommendation:
    primary: AgentPattern
    secondary: AgentPattern | None
    scores: dict[str, float]
    rationale: str
    combos: list[str] = field(default_factory=list)


SIGNAL_WEIGHTS: dict[str, dict[AgentPattern, float]] = {
    "static faq": {AgentPattern.LOOKUP: 10},
    "fixed steps": {AgentPattern.PROMPT_CHAINING: 9},
    "disjoint intents": {AgentPattern.ROUTING: 9},
    "single lookup": {AgentPattern.ONE_SHOT_TOOL: 8, AgentPattern.ROUTING: 4},
    "exploratory tools": {AgentPattern.REACT: 9},
    "reviewable plan": {AgentPattern.PLAN_AND_EXECUTE: 9},
    "long horizon": {AgentPattern.PLAN_AND_EXECUTE: 7},
    "independent checks": {AgentPattern.PARALLELIZATION: 10},
    "multi-domain": {AgentPattern.ORCHESTRATOR_WORKERS: 8, AgentPattern.MULTI_AGENT: 6},
    "numeric rubric": {AgentPattern.EVALUATOR_OPTIMIZER: 9},
    "tone": {AgentPattern.REFLECTION: 8},
    "separate contracts": {AgentPattern.MULTI_AGENT: 9},
}


class PatternSelector:
    def __init__(self, registry: list[PatternProfile]) -> None:
        self.registry = {p.pattern: p for p in registry}

    def score(self, req: RequirementProfile) -> dict[AgentPattern, float]:
        totals: dict[AgentPattern, float] = {p.pattern: 0.0 for p in PATTERN_REGISTRY}
        for signal in req.active_signals():
            for pattern, weight in SIGNAL_WEIGHTS.get(signal, {}).items():
                totals[pattern] += weight
        # Prefer lower complexity when scores tie
        for pattern, prof in self.registry.items():
            if totals[pattern] > 0:
                totals[pattern] -= prof.complexity * 0.1
        # Latency penalty for heavy patterns
        if req.latency_budget_ms < 1500:
            for heavy in (AgentPattern.MULTI_AGENT, AgentPattern.REACT, AgentPattern.PLAN_AND_EXECUTE):
                totals[heavy] -= 3
        return totals

    def recommend(self, req: RequirementProfile) -> PatternRecommendation:
        scores = self.score(req)
        ranked = sorted(scores.items(), key=lambda x: -x[1])
        primary = ranked[0][0]
        secondary = ranked[1][0] if ranked[1][1] > 0 else None
        combos = self._suggest_combos(req, primary)
        rationale = (
            f"Top signals: {req.active_signals()}. "
            f"Primary {primary.value} (score {ranked[0][1]:.1f}). "
            f"Prefer simplest tier that passes eval."
        )
        return PatternRecommendation(
            primary=primary,
            secondary=secondary,
            scores={k.value: round(v, 2) for k, v in scores.items() if v > 0},
            rationale=rationale,
            combos=combos,
        )

    def _suggest_combos(self, req: RequirementProfile, primary: AgentPattern) -> list[str]:
        combos: list[str] = []
        if primary == AgentPattern.ROUTING and req.qualitative_polish:
            combos.append("routing + reflection (specialist then polish)")
        if primary == AgentPattern.PLAN_AND_EXECUTE and req.parallel_checks:
            combos.append("plan-and-execute + parallelization (parallel checks in plan step)")
        if req.numeric_quality_gate and req.qualitative_polish:
            combos.append("evaluator-optimizer then reflection (score gate + tone)")
        if req.multi_domain and primary != AgentPattern.MULTI_AGENT:
            combos.append("orchestrator-workers + shared board")
        return combos


selector = PatternSelector(PATTERN_REGISTRY)
print(pretty(selector.recommend(shopco_return)))

---

## 5. ShopCo Task Bank — Recommendations

In [ ]:
SHOPCO_TASKS: list[RequirementProfile] = [
    RequirementProfile("sc-1", "Support hours FAQ", static_faq=True, latency_budget_ms=500),
    RequirementProfile("sc-2", "Order status by ID", disjoint_intents=True, single_tool=True),
    RequirementProfile("sc-3", "Return + order combined", dynamic_tools=True, qualitative_polish=True),
    RequirementProfile("sc-4", "Billing or shipping unclear", multi_domain=True, separate_agent_roles=True),
]

for t in SHOPCO_TASKS:
    rec = selector.recommend(t)
    print(f"{t.task_id:5} {t.description[:35]:35} → {rec.primary.value}")

---

## 6. ReleaseFlow Task Bank — Recommendations

In [ ]:
RELEASE_TASKS: list[RequirementProfile] = [
    RequirementProfile("rf-1", "Collect changelog + format bullets", fixed_steps=True),
    RequirementProfile("rf-2", "Run CI checks before merge", parallel_checks=True),
    RequirementProfile("rf-3", "Rollout staging→prod with approval", multi_step_sop=True, human_plan_review=True),
    RequirementProfile("rf-4", "Release notes until style score ≥ 80", numeric_quality_gate=True),
    RequirementProfile("rf-5", "Polish announcement tone", qualitative_polish=True),
]

for t in RELEASE_TASKS:
    rec = selector.recommend(t)
    print(f"{t.task_id:5} {t.description[:38]:38} → {rec.primary.value}")

---

## 7. Eval Gate — Escalate Only If Simpler Pattern Fails

In [ ]:
@dataclass
class EvalMetrics:
    accuracy: float
    citation_rate: float
    p95_latency_ms: int

    def passes(self, min_acc: float = 0.85, min_cite: float = 0.8, max_lat: int = 3000) -> bool:
        return self.accuracy >= min_acc and self.citation_rate >= min_cite and self.p95_latency_ms <= max_lat


RETURN_LADDER: list[AgentPattern] = [
    AgentPattern.ROUTING,
    AgentPattern.REACT,
    AgentPattern.MULTI_AGENT,
]

MOCK_EVAL_RETURN: dict[AgentPattern, EvalMetrics] = {
    AgentPattern.ROUTING: EvalMetrics(0.82, 0.75, 400),
    AgentPattern.REACT: EvalMetrics(0.91, 0.92, 1200),
}


def pick_with_eval(ladder: list[AgentPattern], evals: dict[AgentPattern, EvalMetrics]) -> AgentPattern:
    for pattern in ladder:
        m = evals.get(pattern)
        if m and m.passes():
            return pattern
    return ladder[-1]


chosen = pick_with_eval(RETURN_LADDER, MOCK_EVAL_RETURN)
print("Eval-selected pattern:", chosen.value, "(routing failed citation bar → escalated to ReAct)")

---

## 8. Pattern Stubs — Dispatch by Recommendation

In [ ]:
def run_lookup(message: str) -> str:
    for k, v in POLICY_SNIPPETS.items():
        if k in message.lower():
            return v
    return POLICY_SNIPPETS["hours"]


def run_routing(message: str) -> str:
    if "order" in message.lower() or re.search(r"ORD-[0-9]{4}", message, re.I):
        oid = re.search(r"ORD-[0-9]{4}", message.upper())
        return f"Status: {ORDERS_DB.get(oid.group(0) if oid else '', {}).get('status', 'unknown')}"
    return POLICY_SNIPPETS["returns"]


def run_react(message: str) -> str:
    parts = []
    if "return" in message.lower():
        parts.append(POLICY_SNIPPETS["returns"])
    m = re.search(r"ORD-[0-9]{4}", message.upper())
    if m:
        parts.append(f"order={ORDERS_DB[m.group(0)]['status']}")
    return " | ".join(parts)


PATTERN_RUNNERS: dict[AgentPattern, Callable[[str], str]] = {
    AgentPattern.LOOKUP: run_lookup,
    AgentPattern.ROUTING: run_routing,
    AgentPattern.REACT: run_react,
}


def run_recommended(message: str, req: RequirementProfile) -> dict[str, Any]:
    rec = selector.recommend(req)
    runner = PATTERN_RUNNERS.get(rec.primary)
    answer = runner(message) if runner else f"(stub for {rec.primary.value})"
    return {"pattern": rec.primary.value, "answer": answer, "combos": rec.combos}


print(run_recommended("What are support hours?", SHOPCO_TASKS[0]))
print(run_recommended("Return ORD-1001?", shopco_return))

---

## 9. Combination Patterns — When One Is Not Enough

In [ ]:
def routing_plus_reflection(message: str) -> dict[str, Any]:
    draft = run_routing(message)
    issues = []
    if "[pol-" not in draft and "return" in message.lower():
        issues.append("missing citation")
        draft += f" {POLICY_SNIPPETS['returns']}"
    if "unknown" in draft:
        issues.append("weak answer")
        draft += " Please verify your order ID."
    return {"combo": "routing+reflection", "issues_fixed": issues, "answer": draft}


def plan_execute_with_parallel(version: str) -> dict[str, Any]:
    check_results = [{"check": c, "status": "pass"} for c in RELEASE_CHECKS]
    plan = ["parallel_checks", "deploy_staging", "deploy_prod"]
    return {"combo": "plan-and-execute+parallel", "checks": check_results, "plan": plan, "version": version}


print(pretty(routing_plus_reflection("Can I return ORD-1001?")))
print(pretty(plan_execute_with_parallel("2.4.0")))

---

## 10. Comparison Matrix

In [ ]:
def comparison_table() -> list[dict[str, Any]]:
    rows = []
    for p in PATTERN_REGISTRY:
        rows.append({
            "pattern": p.name,
            "complexity": p.complexity,
            "llm_calls": p.typical_llm_calls,
            "top_signal": p.signals[0] if p.signals else "",
            "avoid_when": p.anti_signals[0] if p.anti_signals else "",
        })
    return rows


for row in comparison_table():
    print(f"{row['pattern']:22} complexity={row['complexity']} llm≈{row['llm_calls']} signal={row['top_signal']}")

---

## 11. Architecture Decision Record (ADR) Template

In [ ]:
@dataclass
class PatternADR:
    adr_id: str
    title: str
    context: str
    requirement_profile: RequirementProfile
    recommendation: PatternRecommendation
    eval_evidence: str
    alternatives_rejected: list[str]
    decided_at: str = field(default_factory=utc_now)

    def render(self) -> str:
        return (
            f"# ADR {self.adr_id}: {self.title}\n\n"
            f"**Context:** {self.context}\n\n"
            f"**Decision:** {self.recommendation.primary.value}\n\n"
            f"**Rationale:** {self.recommendation.rationale}\n\n"
            f"**Eval evidence:** {self.eval_evidence}\n\n"
            f"**Rejected:** {', '.join(self.alternatives_rejected)}\n\n"
            f"**Combos considered:** {', '.join(self.recommendation.combos) or 'none'}"
        )


rec = selector.recommend(SHOPCO_TASKS[2])
adr = PatternADR(
    adr_id="ADR-001",
    title="ShopCo return eligibility handler",
    context="Customers ask return questions with order IDs; citations required.",
    requirement_profile=SHOPCO_TASKS[2],
    recommendation=rec,
    eval_evidence="Routing achieved 82% citation rate; ReAct reached 92% in offline eval.",
    alternatives_rejected=["multi_agent (overkill)", "lookup (not static)"],
)
print(adr.render())

---

## 12. Decision Flowchart (Text)

```
START
  │
  ├─ static answer in KB? ──YES──► LOOKUP
  │
  ├─ same steps every time? ──YES──► PROMPT CHAIN
  │
  ├─ disjoint intents? ──YES──► ROUTING (+ reflection if tone matters)
  │
  ├─ one tool enough? ──YES──► ONE-SHOT TOOL
  │
  ├─ parallel independent work? ──YES──► PARALLELIZATION
  │
  ├─ named SOP + plan review? ──YES──► PLAN-AND-EXECUTE
  │
  ├─ unknown tool order? ──YES──► REACT
  ├─ numeric quality bar? ──YES──► EVALUATOR-OPTIMIZER
  ├─ qualitative polish? ──YES──► REFLECTION
  └─ multi-role + QA board? ──YES──► MULTI-AGENT / ORCHESTRATOR-WORKERS

Always: run EVAL on simplest candidate before escalating.
```

---

## 13. Optional Live LLM — Ambiguous Task Classification

Set `USE_LIVE_LLM = True` to classify free-text tasks into `RequirementProfile` hints.

In [ ]:
def infer_signals_live(description: str) -> list[str]:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    resp = llm.invoke([
        SystemMessage(content=(
            "From the task description, return JSON list of applicable signals from: "
            "static faq, fixed steps, disjoint intents, single lookup, exploratory tools, "
            "reviewable plan, independent checks, multi-domain, numeric rubric, tone, separate contracts."
        )),
        HumanMessage(content=description),
    ])
    try:
        return json.loads(str(resp.content))
    except json.JSONDecodeError:
        return []


task = "Run security and unit tests in parallel, then deploy with manager approval"
if USE_LIVE_LLM:
    print("Live signals:", infer_signals_live(task))
else:
    prof = RELEASE_TASKS[2]
    print("Offline signals:", prof.active_signals())

---

## 14. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Skip decision record | Team forgets why ReAct was chosen | Write ADR |
| Ignore latency budget | Slow FAQ path | Penalize heavy patterns in scorer |
| One pattern globally | Wrong tool for release vs support | Per-task `RequirementProfile` |
| No eval escalation | Stuck on failing router | `pick_with_eval` ladder |
| Combo soup | 4 patterns chained | Primary + at most one helper |

---

## 15. Quiz

1. What are the three stages of the selection framework in the intro diagram?
2. Which pattern fits parallel CI checks in ReleaseFlow?
3. When would you combine routing and reflection?
4. What does `pick_with_eval` enforce?
5. What belongs in a Pattern ADR?

---

## 16. Summary

**Choosing the right pattern** is a structured decision: encode requirements as signals, score candidates, prefer the **lowest complexity** winner, validate with **eval**, and document an **ADR**.

ShopCo hours → **lookup**. Order status → **routing**. Return + order → **ReAct** (or routing + reflection). ReleaseFlow checks → **parallelization** inside **plan-and-execute**.

Patterns are tools, not goals. The right pattern is the **simplest one your metrics accept**.